# Frozen neural models
Inference-only PyTorch definitions. Auxiliary training heads are omitted from the exported D checkpoints.

In [ ]:
import numpy as np
import torch
from torch import nn
from tqdm.auto import tqdm


class PlayerBiGRU(nn.Module):
    """Bidirectional player-only D/LIVE classifier."""
    def __init__(self, config):
        super().__init__()
        width, layers = config['hidden_size'], config['num_layers']
        self.gru = nn.GRU(141, width, layers, batch_first=True, bidirectional=True,
                          dropout=config['dropout'] if layers > 1 else 0.)
        self.dropout = nn.Dropout(config['dropout'])
        self.state_head = nn.Linear(2 * width, 2)

    def forward(self, features, valid):
        hidden, _ = self.gru(torch.cat([features, valid.unsqueeze(-1).float()], dim=-1))
        return self.state_head(self.dropout(hidden))


class TemporalResidual(nn.Module):
    """A symmetric dilated convolution with a residual connection."""
    def __init__(self, width, dilation, dropout):
        super().__init__()
        self.conv = nn.Conv1d(width, width, 3, padding=dilation, dilation=dilation)
        self.norm = nn.LayerNorm(width)
        self.dropout = nn.Dropout(dropout)

    def forward(self, hidden):
        update = self.conv(hidden.transpose(1, 2)).transpose(1, 2)
        return hidden + self.dropout(torch.relu(self.norm(update)))


class PlayerTCN(nn.Module):
    """Non-causal player-only D/LIVE classifier."""
    def __init__(self, config):
        super().__init__()
        width = config['hidden_size']
        self.project = nn.Linear(141, width)
        self.blocks = nn.ModuleList([
            TemporalResidual(width, dilation, config['dropout'])
            for dilation in config['tcn_dilations']
        ])
        self.state_head = nn.Linear(width, 2)

    def forward(self, features, valid):
        hidden = self.project(torch.cat([features, valid.unsqueeze(-1).float()], dim=-1))
        for block in self.blocks:
            hidden = block(hidden) * valid.unsqueeze(-1)
        return self.state_head(hidden)


class PossessionEncoder(nn.Module):
    """Team-wise set pooling followed by the frozen SkillCorner GRU."""
    def __init__(self, hidden_size=48):
        super().__init__()
        self.player_encoder = nn.Sequential(nn.Linear(6, 24), nn.ReLU(), nn.Linear(24, 24), nn.ReLU())
        self.temporal = nn.GRU(101, hidden_size, batch_first=True)
        self.head = nn.Linear(hidden_size, 2)

    def forward(self, positions, observed, ball, ball_observed):
        ball_valid = ball_observed[..., None, None, None]
        relative = (positions - ball[..., None, None, :]) * ball_valid
        distance = torch.linalg.vector_norm(relative, dim=-1, keepdim=True)
        tokens = torch.cat([positions / 50, relative / 20, distance / 20,
                            ball_valid.expand(*positions.shape[:-1], 1)], dim=-1)
        encoded = self.player_encoder(tokens)
        mask = observed[..., None]
        mean = (encoded * mask).sum(-2) / mask.sum(-2).clamp_min(1)
        maximum = encoded.masked_fill(~mask, -1e4).max(-2).values
        maximum = torch.where(mask.any(-2), maximum, torch.zeros_like(maximum))
        counts = observed.sum(-1).float() / 11
        spatial = torch.cat([mean.flatten(-2), maximum.flatten(-2), counts,
                             ball / 50, ball_observed[..., None].float()], dim=-1)
        hidden, _ = self.temporal(spatial)
        return hidden


class PossessionBiGRU(nn.Module):
    """A residual correction to the frozen encoder's H/A log probabilities."""
    def __init__(self, config):
        super().__init__()
        self.gru = nn.GRU(51, config['hidden_size'], config['num_layers'],
                          batch_first=True, bidirectional=True,
                          dropout=config['dropout'] if config['num_layers'] > 1 else 0.)
        self.dropout = nn.Dropout(config['dropout'])
        self.output = nn.Linear(2 * config['hidden_size'], 2)

    def forward(self, features, valid):
        inputs = torch.cat([features, valid[..., None].to(features.dtype)], dim=-1)
        hidden, _ = self.gru(inputs)
        correction = self.output(self.dropout(hidden))
        return torch.log(features[..., -2:].clamp_min(1e-7)) + correction


## Windowed inference
All models run on CPU. Model weights, preprocessing and temporal context are fixed.

In [ ]:
def centered_windows(data, output_frames):
    return [(low, start, min(start + output_frames, high), high)
            for low, high in input_blocks(data)
            for start in range(low, high, output_frames)]


def centered_window(features, item, context, output_frames, possession=False):
    low, start, end, high = item
    begin = max(low, start - context)
    finish = min(high, start + output_frames + context)
    offset = context - (start - begin)
    values = np.zeros((2 * context + output_frames, features.shape[1]), dtype=np.float32)
    if possession:
        values[:, -2:] = .5
    valid = np.zeros(len(values), dtype=bool)
    values[offset:offset + finish - begin] = features[begin:finish]
    valid[offset:offset + finish - begin] = True
    return values, valid


@torch.no_grad()
def predict_centered(model, features, data, config, batch_size, possession=False):
    """Predict only window centres; context and padding match the trained model."""
    model.eval()
    items = centered_windows(data, config['output_frames'])
    probability = np.zeros((len(features), 2), dtype=np.float32)
    batches = range(0, len(items), batch_size)
    for start in tqdm(batches, desc=type(model).__name__):
        group = items[start:start + batch_size]
        arrays = [centered_window(features, item, config['context_frames'],
                                  config['output_frames'], possession) for item in group]
        x = torch.from_numpy(np.stack([a[0] for a in arrays]))
        valid = torch.from_numpy(np.stack([a[1] for a in arrays]))
        values = model(x, valid).softmax(-1).numpy()
        for row, (_, first, last, _) in enumerate(group):
            offset = config['context_frames']
            probability[first:last] = values[row, offset:offset + last - first]
    return probability


@torch.no_grad()
def predict_encoder(model, data, batch_size):
    """Reproduce 035: 40 past frames and 40 output frames, with zero left padding."""
    model.eval()
    items = [(low, start, min(start + 40, high))
             for low, high in input_blocks(data) for start in range(low, high, 40)]
    embedding = np.zeros((len(data['frame']), 48), dtype=np.float32)
    keys = ('positions', 'observed', 'ball', 'ball_observed')
    for offset in tqdm(range(0, len(items), batch_size), desc='PossessionEncoder'):
        group = items[offset:offset + batch_size]
        windows = []
        for low, start, end in group:
            begin = max(low, start - 40)
            padding = 40 - (start - begin)
            window = {}
            for key in keys:
                value = data[key]
                result = np.zeros((80,) + value.shape[1:], dtype=value.dtype)
                result[padding:padding + end - begin] = value[begin:end]
                window[key] = result
            windows.append(window)
        tensors = {key: torch.from_numpy(np.stack([w[key] for w in windows])) for key in keys}
        hidden = model(**tensors).numpy()
        for row, (_, start, end) in enumerate(group):
            embedding[start:end] = hidden[row, 40:40 + end - start]
    # Compute the original head on saved embeddings, matching the frozen 064 input path.
    base_probability = model.head(torch.from_numpy(embedding)).softmax(-1).numpy()
    return embedding, base_probability


## Temporal decoding
The duration model and H/A switch penalty were selected using external data. No reference labels are read here.

In [ ]:
def boolean_runs(mask):
    edges=np.flatnonzero(np.diff(np.r_[False,mask,False].astype(np.int8)))
    return list(zip(edges[::2],edges[1::2]))

def viterbi(probability,penalty):
    # Two-state H/A chain, no forced minimum duration; future evidence allowed.
    logp=np.log(np.clip(probability,1e-7,1.));n=len(logp)
    if n==0:return np.empty(0,np.int8)
    if penalty==0:return probability.argmax(1).astype(np.int8)
    score=logp[0].copy();back=np.zeros((n,2),np.int8)
    for t in range(1,n):
        candidates=np.array([[score[0],score[1]-penalty],[score[0]-penalty,score[1]]])
        back[t]=candidates.argmax(1);score=candidates.max(1)+logp[t]
    result=np.empty(n,np.int8);result[-1]=score.argmax()
    for t in range(n-1,0,-1):result[t-1]=back[t,result[t]]
    return result

def decode_ha(probability,data,penalty,live=None):
    labels=probability.argmax(1).astype(np.int8)
    for lo,hi in input_blocks(data):
        spans=[(0,hi-lo)] if live is None else boolean_runs(live[lo:hi])
        for a,b in spans:labels[lo+a:lo+b]=viterbi(probability[lo+a:lo+b],penalty)
    return labels

def age_viterbi(emissions, hazards, weight, return_score=False):
    # Exact Viterbi in a finite age-expanded chain; final state is right-censored.
    n, states = emissions.shape; ages = hazards.shape[1]
    log_stay, log_end = weight*np.log1p(-hazards), weight*np.log(hazards)
    # Stationary age distribution handles unknown age at the left boundary.
    survival = np.cumprod(np.c_[np.ones(2), 1-hazards[:,:-1]],axis=1)
    survival[:,-1] /= hazards[:,-1]
    prior = survival / survival.sum(axis=1,keepdims=True) / 2
    scores = weight*np.log(prior) + emissions[0,:,None]
    switch_age = np.zeros((n,2),np.int16)
    tail_from_cap = np.zeros((n,2),bool)
    for t in range(1,n):
        ended = scores + log_end
        origins = ended.argmax(axis=1)
        nxt = np.full_like(scores,-np.inf)
        for state in [0,1]:
            switch_age[t,state] = origins[1-state]
            nxt[state,0] = ended[1-state,origins[1-state]]
        stayed = scores + log_stay
        nxt[:,1:] = stayed[:,:-1]
        tail_from_cap[t] = stayed[:,-1] > stayed[:,-2]
        nxt[:,-1] = np.maximum(stayed[:,-2],stayed[:,-1])
        scores = nxt + emissions[t,:,None]
    state, age = np.unravel_index(scores.argmax(),scores.shape)
    best_score = float(scores[state,age]); labels = np.empty(n,np.int8)
    for t in range(n-1,-1,-1):
        labels[t] = state
        if t == 0: break
        if age == 0:
            age = int(switch_age[t,state]); state = 1-state
        elif age == ages-1 and tail_from_cap[t,state]: pass
        else: age -= 1
    return (labels,best_score) if return_score else labels

def duration_decode(probability, data, weight, duration_config, hazards):
    out = np.zeros(len(probability),np.float32)
    assert len(probability) == len(data["frame"])
    for lo,hi in input_blocks(data):
        p = np.clip(probability[lo:hi],1e-6,1-1e-6)
        logp = np.stack([np.log1p(-p),np.log(p)],axis=1)
        starts = np.arange(0,len(p),duration_config["stride"])
        counts = np.minimum(duration_config["stride"],len(p)-starts)
        # Mean evidence per 0.5-second bin prevents counting correlated frames 5 times.
        emission = np.add.reduceat(logp,starts,axis=0)/counts[:,None]
        labels = age_viterbi(emission,hazards,weight)
        out[lo:hi] = np.repeat(labels,counts)
    return out
